In [ ]:
# %% [code] {"execution":{"iopub.status.busy":"2025-09-18T21:11:47.859655Z","iopub.execute_input":"2025-09-18T21:11:47.860039Z","iopub.status.idle":"2025-09-18T21:11:47.878935Z","shell.execute_reply.started":"2025-09-18T21:11:47.860016Z","shell.execute_reply":"2025-09-18T21:11:47.878099Z"}}
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split,RandomizedSearchCV, KFold,  cross_val_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import optuna
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
train_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
train_df.head()

In [ ]:
coor_mat = train_df.corr(numeric_only=True)
sns.heatmap(coor_mat, cmap="YlGnBu", annot=True)

threshold = 0.5
mask = (np.abs(coor_mat) >= threshold)

filtered_coor_mat = coor_mat.where(mask)

plt.figure(figsize=(20, 15))
sns.heatmap(filtered_coor_mat, cmap="YlGnBu", annot=True, fmt=".2f")
plt.show()

In [ ]:
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    if confusion_matrix.size == 0:  # no data
        return np.nan
    chi2 = chi2_contingency(confusion_matrix, correction=False)[0]
    n = confusion_matrix.sum().sum()
    if n == 0:
        return np.nan

    phi2 = chi2 / n
    r, k = confusion_matrix.shape

    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)

    denom = min(kcorr-1, rcorr-1)
    if denom <= 0:   # degenerate case: cannot divide
        return np.nan

    return np.sqrt(phi2corr / denom)

target = "SalePrice"
predictors = [c for c in train_df.columns if c != target]

df_cat = train_df.copy()
for col in predictors + [target]:
    df_cat[col] = df_cat[col].astype("category")


results = {}
for col in predictors:
    results[col] = cramers_v(df_cat[target], df_cat[col])

results_series = pd.Series(results).sort_values(ascending=False)
print("Cramér's V between target and predictors (sorted):\n")
print(results_series.head(10))

#Adding flags
train_df['HasPool'] = (train_df['PoolArea'] > 0).astype(int)
train_df['HasGarage'] = (train_df['GarageArea'] > 0).astype(int)
train_df['HasBsmt'] = (train_df['TotalBsmtSF'] > 0).astype(int)
train_df['HasFireplace'] = (train_df['Fireplaces'] > 0).astype(int)
train_df['Has2ndFlr'] = (train_df['2ndFlrSF'] > 0).astype(int)

selected_features = ["FullBath", "ExterQual", "KitchenQual", "OverallQual", "GrLivArea", "TotalBsmtSF", "1stFlrSF", "GarageCars", "GarageArea", "HasPool" , "HasGarage", "HasBsmt" , "HasFireplace","Has2ndFlr", "Neighborhood" , "YearBuilt", "MSZoning","FireplaceQu" , "PoolQC", "SaleType"]
X = train_df[selected_features]
Y = train_df["SalePrice"]

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class QualityMapper(BaseEstimator, TransformerMixin):
    def __init__(self, mapping):
        self.mapping = mapping

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = np.array(X, copy=True)

        for col in range(X_copy.shape[1]):
            unique_vals = np.unique(X_copy[:, col])
            mapping_dict = {val: self.mapping.get(val, val) for val in unique_vals}
            X_copy[:, col] = np.vectorize(mapping_dict.get)(X_copy[:, col])

        return X_copy.astype(float)


def train_xgboost_optuna(X_df, y_series, test_size=0.2, random_state=42, eval_model=True, n_trials=200):

    qual_mapping = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "NA": 0}
    
    ordinal_cols = [
        "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", 
        "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", 
        "GarageCond", "PoolQC"
    ]
    ordinal_cols = [col for col in ordinal_cols if col in X_df.columns]

    nominal_cols = ['Neighborhood']

    numerical_features = X_df.select_dtypes(include=['int64', 'float64']).columns
    processed_cols = ordinal_cols + nominal_cols
    numerical_features = [col for col in numerical_features if col not in processed_cols]

    numerical_transformer = SimpleImputer(strategy='median')
    
    quality_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ('mapper', QualityMapper(qual_mapping))
    ])

    nominal_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_features),
            ('qual', quality_transformer, ordinal_cols),
            ('nom', nominal_transformer, nominal_cols)  
        ],
        remainder='drop'
    )

    model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                     ('regressor', xgb.XGBRegressor(random_state=random_state))])

    X = X_df
    y = np.log1p(y_series.to_numpy()) 
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    def objective(trial):
        params = {
            'regressor__n_estimators': trial.suggest_int('n_estimators', 500, 2000),
            'regressor__max_depth': trial.suggest_int('max_depth', 3, 12),
            'regressor__learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'regressor__subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'regressor__colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'regressor__min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'regressor__gamma': trial.suggest_float('gamma', 0.0, 5.0),
            'regressor__reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
            'regressor__reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0)
        }
        
        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=random_state)
        model_pipeline.set_params(**params)
        ##model_pipeline.fit(X_tr, y_tr)
        ##preds = model_pipeline.predict(X_val)
        ##rmse = np.sqrt(mean_squared_error(y_val, preds))
        kFold = KFold(n_splits = 5, random_state = random_state, shuffle= True)
        rmse = cross_val_score(
            model_pipeline, X_train, y_train,
            scoring="neg_root_mean_squared_error",
            cv=kFold, n_jobs=-1            
        )
        rmse = rmse.mean()
        return rmse

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    print("Best parameters:", study.best_trial.params)

    best_params = {key.replace('regressor__', ''): value for key, value in study.best_trial.params.items()}
    best_model = xgb.XGBRegressor(**best_params)

    final_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                     ('regressor', best_model)])

    final_pipeline.fit(X_train, y_train)

    if eval_model:
        y_pred_log = final_pipeline.predict(X_test)
        y_pred = np.expm1(y_pred_log)
        test_rmse = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred))
        print(f"\nTest RMSE: {test_rmse:.2f}")

    return final_pipeline, study

train_xgboost_optuna(X,Y)

In [ ]:
Y = np.log1p(train_df["SalePrice"])
print(Y)